# SAT Experiment + Table Rendering


In [ ]:
import csv
import itertools
import json
import sys

import jax
import jax.numpy as jnp
import numpy as np
import optax
from flax.training import train_state
from jax import random
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

sys.path.insert(0, '/home/blohmp1/diffusion-composition/experiments/new_sat')
from sat.score_composer import compose_formula_with_reference
from sat.fuzzy_checker import analyze_sampler_sparse
from sat.propositional_diffusion_models import HardcodedScore

seed = 0
key = random.PRNGKey(seed)
bs = 1024

beta_0 = 0.1
beta_1 = 20.0
log_alpha = lambda t: -0.5*t*beta_0-0.25*t**2*(beta_1-beta_0)
log_sigma = lambda t: jnp.log(t)
dlog_alphadt = jax.grad(lambda t: log_alpha(t).sum())
beta = lambda t: (1 + 0.5*t*beta_0 + 0.5*t**2*(beta_1-beta_0))

def majority_dimacs(ndim: int, k: int = None):
    if k is None:
        k = ndim // 2 + 1
    return [[c + 1 for c in comb] for comb in itertools.combinations(range(ndim), k)]

def xor_dimacs(n):
    clauses = []
    for bits in itertools.product([0,1], repeat=n):
        if sum(bits) % 2 == 0:
            clause = [-(i+1) if bits[i] else (i+1) for i in range(n)]
            clauses.append(clause)
    return clauses

def exactly_one_dimacs(ndim: int):
    return [[i + 1 for i in range(ndim)]] + [[-(i + 1), -(j + 1)] for i in range(ndim) for j in range(i + 1, ndim)]

def propositional_Gausses(ndim, sigma=.5, key=key):
    model = HardcodedScore(num_out=ndim, sigma=sigma*1.5, which='all', axis=0)
    key, init_key = random.split(key)
    optimizer = optax.adam(learning_rate=2e-4)
    state = train_state.TrainState.create(
        apply_fn=model.apply,
        params=model.init(init_key, np.ones([bs, 1]), np.zeros([bs, ndim])),
        tx=optimizer,
    )
    base_model = state
    dim_models = []
    for i in range(ndim):
        model = HardcodedScore(num_out=ndim, sigma=sigma, which='one_side', axis=i, sign=1)
        key, init_key = random.split(key)
        optimizer = optax.adam(learning_rate=2e-4)
        state = train_state.TrainState.create(
            apply_fn=model.apply,
            params=model.init(init_key, np.ones([bs, 1]), np.zeros([bs, ndim])),
            tx=optimizer,
        )
        dim_models.append(state)
    return base_model, dim_models

# Run SAT experiment and generate CSV result file

In [ ]:
# these should be params
bs = 100 #number of particles
sigma = .5
ndims = [2,5]#[2, 5, 10]
lambs = [1,100] #lambda
out_csv = 'sat_results_combined.csv'

# these should be static
dt = 1e-2 #delta t
t_init = 1.0 #initial t
n = int(t_init / dt) #number timesteps


with open(out_csv, 'w', encoding='utf-8') as f:
    print('method;lambda;instance;d;n_points;n_corr;n_outside;n_wrong;perplexity;num_modes;max_count;frac_corr_all;frac_corr_within;x_gen_init;x_gen_final', file=f)

    for ndim in ndims: #[2,5,10,1,3,4,6,7,8,9]:
        for name,instance in [('majority',majority_dimacs(ndim)),
                              ('xor',xor_dimacs(ndim)),
                              ('exactly_one',exactly_one_dimacs(ndim))]:
            for method in ['dombi', 'prob']:
                for lamb in lambs: #[0.5,1,5,10,100]:
                    if method == 'prob' and not lamb == 1:
                        continue

                    t = t_init * jnp.ones((bs, 1))
                    key, ikey = random.split(key, num=2)
                    x_gen = jnp.zeros((bs, n + 1, ndim))
                    x_gen = x_gen.at[:, 0, :].set(random.normal(ikey, shape=(bs, ndim)))

                    base, dim_models = propositional_Gausses(ndim, sigma=sigma)
                    for index in range(n):
                        x_t = x_gen[:, index, :]
                        key, ikey = random.split(key, num=2)
                        (scores, _), _, _, _ = compose_formula_with_reference(
                            dim_models, t, x_t, instance, base, ell=lamb, method=method, gamma=1
                        )

                        dx = -dt * (dlog_alphadt(t) * x_t - 2 * beta(t) * scores)
                        dx += jnp.sqrt(2 * jnp.exp(log_sigma(t)) * beta(t) * dt) * random.normal(ikey, shape=(bs, ndim))
                        x_gen = x_gen.at[:, index + 1, :].set(x_t + dx)
                        t -= dt

                    x = analyze_sampler_sparse(x_gen[:, -1, :], instance, 2, .75, metric='linf')
                    anycorr = x.correct_modes_uniformity

                    if ndim == 2:
                        x_gen_init = json.dumps(np.asarray(x_gen[:, 0, :]).tolist(), separators=(',', ':'))
                        x_gen_final = json.dumps(np.asarray(x_gen[:, -1, :]).tolist(), separators=(',', ':'))
                    else:
                        x_gen_init = ''
                        x_gen_final = ''

                    print(
                        method,
                        lamb,
                        name,
                        x.d,
                        x.n_points,
                        x.correct_modes_uniformity.n if anycorr else 0,
                        x.n_points - x.n_within_sigma,
                        x.n_within_sigma - x.correct_modes_uniformity.n if anycorr else 0,
                        x.correct_modes_uniformity.perplexity if anycorr else 0,
                        x.correct_modes_uniformity.K if anycorr else 0,
                        max(x.correct_modes_uniformity.counts) if anycorr else 0,
                        x.correct_modes_uniformity.n / x.n_points if anycorr else 0,
                        x.correct_modes_uniformity.n / x.n_within_sigma if anycorr else 0,
                        x_gen_init,
                        x_gen_final,
                        sep=';',
                        file=f
                    )

print(f'Wrote results to {out_csv}')

## Render Table

In [ ]:
import csv
from collections import defaultdict

# read CSV
input_csv = out_csv if 'out_csv' in globals() else 'sat_results_combined.csv'
rows = []
with open(input_csv) as f:
    reader = csv.DictReader(f, delimiter=';')
    for row in reader:
        row['d'] = int(row['d'])
        row['n_points'] = int(row['n_points'])
        row['n_outside'] = int(row['n_outside'])
        row['frac_corr_within'] = float(row['frac_corr_within'])
        row['perplexity'] = float(row['perplexity'])
        row['method'] = row['method'].lower()
        rows.append(row)

# target setup
dims = [2, 5, 10]
methods = {'prob': 'Prod', 'dombi': 'Dombi'}
formulas = [('$\\text{Maj}_k$','majority'), ('$\\text{XOR}_k$','xor'), ('$\\text{OneHot}_k$','exactly_one')]

# collect data: {formula: {method: {d: (corr_within, one_minus_outside, perplexity)}}}
table_data = defaultdict(lambda: defaultdict(dict))
for r in rows:
    if r['d'] not in dims:
        continue
    if r['method'] == 'dombi' and float(r['lambda']) != 100:
        continue
    if r['method'] == 'prob' and float(r['lambda']) != 1:
        continue

    corr_within = r['frac_corr_within']
    one_minus_outside = 1 - (r['n_outside'] / r['n_points'])
    perplexity = r['perplexity']
    modes = r['num_modes']

    table_data[r['instance']][methods[r['method']]][r['d']] = (
        corr_within, one_minus_outside, perplexity
    )

# --- Generate LaTeX ---
print('\\begin{table*}[h]')
print('\\centering')
print('\\begin{tabular}{cc'+''.join(['ccc' for _ in dims]) + '}')
print('\\toprule')

# header row
print('&&' + '&'.join([f'\\multicolumn{{3}}{{c}}{{Dim {d}}}' for d in dims]) + ' \\\\')
print('' + ''.join([f'\\cmidrule(l{{3pt}}r{{3pt}}){{{col*3+3}-{col*3+5}}}' for col,_ in enumerate(dims)]) + ' \\\\')
print('Formula&Method&' + '&'.join(['SAT&Surv&P' for _ in dims]) + ' \\\\')
print('\\midrule')

# body
for name, formula in formulas:
    methods_here = list(table_data[formula].keys())
    if not methods_here:
        continue
    print(f'\\multirow{{{len(methods_here)}}}{{*}}{{{name}}}', end=' ')
    for index, method in enumerate(methods_here):
        if index > 0:
            print(' &', end=' ')
        else:
            print('&', end=' ')
        print(method, end=' ')
        for d in dims:
            vals = table_data[formula][method].get(d, ('-', '-', '-'))
            if isinstance(vals[0], float):
                print(f'&{vals[0]:.2f}&{vals[1]:.2f}&{vals[2]:.2f}', end=' ')
            else:
                print('& - & - & -', end=' ')
        print('\\\\')
    print('\\midrule')

print('\\bottomrule')
print('\\end{tabular}')
print('\\caption{Comparison of Dombi vs Product across selected dimensions.}')
print('\\end{table*}')


## Optional Plot Code (from CSV, d=2)

This cell loads the experiment CSV and plots the d=2 results (SAT / survival / perplexity) for each formula and method.
For results like in the paper run simulation with gamma=.5, is an optical adjustment for this figure.

In [ ]:
from matplotlib import patches

rows_2d = []
with open(out_csv, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f, delimiter=';')
    for row in reader:
        if int(row['d']) == 2:
            rows_2d.append(row)

points = {}
x_gen = None

for row in rows_2d:
    method = row['method']
    lamb = float(row['lambda'])
    instance = row['instance']

    if method == 'prob' and lamb != 1:
        continue
    if method == 'dombi' and lamb != 100:
        continue

    if row.get('x_gen_final'):
        points[instance, method] = np.array(json.loads(row['x_gen_final']))
    if x_gen is None and row.get('x_gen_init'):
        x_gen = np.array(json.loads(row['x_gen_init']))[:, None, :]

if x_gen is None:
    raise ValueError('No x_gen_init found in CSV. Re-run the SAT simulation cell first.')

required_keys = [('majority', 'dombi'), ('majority', 'prob'), ('xor', 'dombi'), ('xor', 'prob')]
missing = [k for k in required_keys if k not in points]
if missing:
    raise ValueError(f'Missing point clouds in CSV for: {missing}. Re-run the SAT simulation cell first.')

plt.figure(figsize=(6, 3))

plt.subplot(1,2,1)
plt.grid()
plt.scatter(x_gen[:,0,0], x_gen[:,0,1], label='initial', alpha=.3, color="green")
plt.scatter(points["majority","dombi"][:,0], points["majority","dombi"][:,1], label='Dombi', alpha=.3, color="#0090D4")
plt.scatter(points["majority","prob"][:,0], points["majority","prob"][:,1], label='Product', alpha=.3, color="red")
plt.xlim(-4, 4)
plt.ylim(-4, 4)
plt.title("$P_1 \\vee P_2$")
plt.legend()
rect = patches.Rectangle((2-0.75, 2-0.75), 1.5, 1.5,
                         linewidth=2, edgecolor='black', facecolor='none')
plt.gca().add_patch(rect)
rect = patches.Rectangle((2-0.75, -2-0.75), 1.5, 1.5,
                         linewidth=2, edgecolor='black', facecolor='none')
plt.gca().add_patch(rect)
rect = patches.Rectangle((-2-0.75, 2-0.75), 1.5, 1.5,
                         linewidth=2, edgecolor='black', facecolor='none')
plt.gca().add_patch(rect)


plt.subplot(1,2,2)
plt.grid()
plt.scatter(x_gen[:,0,0], x_gen[:,0,1], label='initial', alpha=.3, color="green")
plt.scatter(points["xor","dombi"][:,0], points["xor","dombi"][:,1], label='Dombi', alpha=.3, color="#0090D4")
plt.scatter(points["xor","prob"][:,0], points["xor","prob"][:,1], label='Product', alpha=.3, color="red")
plt.xlim(-4, 4)
plt.ylim(-4, 4)
plt.legend()
plt.title("$P_1 \\text{ XOR } P_2$")

rect = patches.Rectangle((2-0.75, -2-0.75), 1.5, 1.5,
                         linewidth=2, edgecolor='black', facecolor='none')
plt.gca().add_patch(rect)
rect = patches.Rectangle((-2-0.75, 2-0.75), 1.5, 1.5,
                         linewidth=2, edgecolor='black', facecolor='none')
plt.gca().add_patch(rect)

# plt.savefig("R2_formulas.pdf", bbox_inches='tight', pad_inches=0)
plt.show()
print(xor_dimacs(2))